# 03 · Foundation component: CLIP inside the system

In [ ]:
import os, sys
REPO_URL = "https://github.com/YUPWE/car-models.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/car-models"
    if not os.path.exists(ROOT):
        !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
    !git pull -q
    !pip install -q imagehash==4.3.2
    os.environ["CARS_DATA_DIR"] = "/content/cars_data"
else:
    ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.makedirs("results", exist_ok=True)
os.makedirs("weights", exist_ok=True)
QUICK = os.environ.get("QUICK", "0") == "1"
print("Working in", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
import json, numpy as np, torch, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, log_loss
from src import data
from src.data import CarDataset, get_transforms
from src.model import build_resnet18, predict_probs, get_device
from src.clip_gate import ClipZeroShot, load_pil
from src.system import save_config

data.download()
device = get_device()
names = json.load(open("results/model_names.json"))
val_df = data.load_split("val")
if QUICK: val_df = val_df.sample(64, random_state=0)

resnet = build_resnet18(freeze_backbone=False, pretrained=False, num_classes=196)
resnet.load_state_dict(torch.load("weights/resnet18_finetuned.pt", map_location=device)); resnet.to(device)
p_res, y_val = predict_probs(resnet, DataLoader(CarDataset(val_df, get_transforms(False)), batch_size=128), device)

clip = ClipZeroShot(names, device=device)
p_clip, oos_val = clip(load_pil(f"{data.IMG_DIR}/{p}" for p in val_df.path))
for name, p in [("CLIP zero-shot (no training)", p_clip), ("ResNet-18 fine-tuned", p_res)]:
    pred = p.argmax(1)
    print(f"{name:30s} val acc {accuracy_score(y_val, pred):.3f} | macro-F1 {f1_score(y_val, pred, average='macro'):.3f}")

## Second opinion: choose the mixing weight w

In [ ]:
def mixed(w):
    m = np.clip(w * p_res + (1 - w) * p_clip, 1e-9, 1)
    return m / m.sum(1, keepdims=True)

ws = np.round(np.linspace(0, 1, 21), 2)
losses = [log_loss(y_val, mixed(w), labels=range(196)) for w in ws]
accs = [accuracy_score(y_val, mixed(w).argmax(1)) for w in ws]
best_w = float(ws[int(np.argmin(losses))])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ws, losses, "o-"); ax[0].axvline(best_w, ls="--", c="k"); ax[0].set_title("Validation log-loss vs w (lower is better)")
ax[1].plot(ws, accs, "o-"); ax[1].axvline(best_w, ls="--", c="k"); ax[1].set_title("Validation accuracy vs w")
for a in ax: a.set_xlabel("w  (0 = CLIP only, 1 = ResNet only)"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/mixing_weight.png", dpi=120); plt.show()
print("chosen w =", best_w, "| val accuracy at w:", round(accs[int(np.argmin(losses))], 3), "| ResNet only:", round(accs[-1], 3))

## Not-a-car gate

In [ ]:
grid = np.round(np.arange(0.30, 1.0, 0.05), 2)
false_reject = np.array([(oos_val >= t).mean() for t in grid])
ok = grid[false_reject <= 0.01]
oos_threshold = float(ok.min()) if len(ok) else 0.95
print(f"chosen not-a-car threshold = {oos_threshold} -> rejects {(oos_val >= oos_threshold).mean():.1%} of validation car photos")

from skimage import data as skd
from PIL import Image
ood = ["astronaut", "chelsea", "coffee", "rocket", "camera", "coins", "moon", "page", "text",
       "clock", "horse", "hubble_deep_field", "immunohistochemistry", "retina", "brick", "grass", "gravel", "colorwheel"]
def to_pil(a):
    a = np.asarray(a)
    if a.dtype == bool: a = a.astype(np.uint8) * 255
    if a.dtype != np.uint8: a = (255 * (a - a.min()) / (a.max() - a.min() + 1e-9)).astype(np.uint8)
    return Image.fromarray(a).convert("RGB")
_, oos_ood = clip([to_pil(getattr(skd, n)()) for n in ood])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(oos_val, bins=20, range=(0, 1), alpha=.7, label="validation car photos")
ax.hist(oos_ood, bins=20, range=(0, 1), alpha=.7, label="non-car photos")
ax.axvline(oos_threshold, ls="--", c="k", label="chosen threshold"); ax.set_xlabel("CLIP p(not a car)"); ax.legend()
plt.tight_layout(); plt.savefig("results/not_a_car_gate.png", dpi=120); plt.show()
print(f"non-car photos rejected: {(oos_ood >= oos_threshold).sum()} / {len(ood)}")
for n, s in zip(ood, oos_ood): print(f"  {n:22s} p(not a car) = {s:.2f} {'-> rejected' if s >= oos_threshold else '-> NOT rejected'}")
print(save_config(w=best_w, oos_threshold=oos_threshold, clip_model=clip.model.name_or_path))